**CMD + MIST isochrones (ages in Gyr)**

In [2]:
#MIST isochrones
# install if needed 
!pip install -q isochrones  

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from isochrones import get_ichrone

#load data
STAR_FILE = "final_reordered.csv"
df = pd.read_csv(STAR_FILE, low_memory=False)

# cluster name
cluster_col = next((c for c in df.columns if "cluster" in c.lower()), None)
if cluster_col is None:
    raise ValueError("Could not find a cluster column in final_reordered.csv")

mask_ca1 = df[cluster_col].astype(str).str.contains("Casado-Alessi_1", case=False, na=False)
df_ca1 = df.loc[mask_ca1].copy()
print(f"Casado-Alessi 1 stars found: {len(df_ca1)}")

# helper to pick columns 
def pick(col_patterns):
    cols = df_ca1.columns
    cols_lower = cols.str.lower()
    for col, cl in zip(cols, cols_lower):
        if all(p.lower() in cl for p in col_patterns):
            return col
    return None

# gaia mags
g_col  = pick(["gmag"]) or pick(["phot_g"]) or pick(["g_mean_mag"])
bp_col = pick(["bp", "mag"])
rp_col = pick(["rp", "mag"])
if g_col is None or bp_col is None or rp_col is None:
    raise ValueError(f"Could not find G/BP/RP columns: g={g_col}, bp={bp_col}, rp={rp_col}")

# astrometry columns
plx_col   = pick(["plx"]) or pick(["parallax"])
pmra_col  = pick(["pmra"])
pmdec_col = pick(["pmde"]) or pick(["pmdec"])
if plx_col is None or pmra_col is None or pmdec_col is None:
    raise ValueError(f"Could not find astrometry columns: plx={plx_col}, pmra={pmra_col}, pmdec={pmdec_col}")

# colors
df_ca1["BP_RP"] = df_ca1[bp_col] - df_ca1[rp_col]
color_cl = df_ca1["BP_RP"].to_numpy()
g_mag_cl = pd.to_numeric(df_ca1[g_col], errors="coerce").to_numpy()
good = np.isfinite(color_cl) & np.isfinite(g_mag_cl)

print(f"→ Plotted {good.sum()} stars from Casado-Alessi 1")

# define x stars
def robust_stats(series):
    s = pd.to_numeric(series, errors="coerce")
    med = np.nanmedian(s)
    mad = np.nanmedian(np.abs(s - med))
    sigma = 1.4826 * mad if mad > 0 else np.nan
    return med, sigma

# cluster stats
med_plx,  sig_plx  = robust_stats(df_ca1[plx_col])
med_pmra, sig_pmra = robust_stats(df_ca1[pmra_col])
med_pmde, sig_pmde = robust_stats(df_ca1[pmdec_col])

df_ca1["dPlx"]  = df_ca1[plx_col]  - med_plx
df_ca1["dpmRA"] = df_ca1[pmra_col] - med_pmra
df_ca1["dpmDE"] = df_ca1[pmdec_col]- med_pmde

df_ca1["zPlx"]  = df_ca1["dPlx"]  / sig_plx  if np.isfinite(sig_plx)  and sig_plx  > 0 else np.nan
df_ca1["zpmRA"] = df_ca1["dpmRA"] / sig_pmra if np.isfinite(sig_pmra) and sig_pmra > 0 else np.nan
df_ca1["zpmDE"] = df_ca1["dpmDE"] / sig_pmde if np.isfinite(sig_pmde) and sig_pmde > 0 else np.nan

z_stack = np.vstack([
    np.abs(df_ca1["zPlx"].to_numpy()),
    np.abs(df_ca1["zpmRA"].to_numpy()),
    np.abs(df_ca1["zpmDE"].to_numpy())
])
max_abs_z = np.nanmax(z_stack, axis=0)

# "Bad" definition: 10 <= G <= 11 AND max|z| >= 4
slice_10_11 = (g_mag_cl >= 10.0) & (g_mag_cl <= 11.0)
bad_mask = slice_10_11 & np.isfinite(max_abs_z) & (max_abs_z >= 4.0)

print(f"Stars with {g_col} between 10 and 11: {slice_10_11.sum()}")
print(f'  → "Bad" (max|z| ≥ 4): {bad_mask.sum()}')

# MIST isochrones
mist = get_ichrone('mist', bands=['G', 'BP', 'RP'])   

# Gaia DR3-like extinction coefficients
K_G  = 2.3  # A_G  / E(B-V)
K_BP = 2.9  # A_BP / E(B-V)
K_RP = 2.0  # A_RP / E(B-V)

EBV_GLOBAL = 0.03 
DM_GLOBAL  = 9  

ages_gyr = [0.8, 1.45, 2.0, 2.5, 2.75, 3.0, 3.5]   
feh = -0.22                                      

plt.figure(figsize=(9, 8))

# Plot all targets
plt.scatter(color_cl[good], g_mag_cl[good], s=15, color='gray',
            edgecolors='k', linewidth=0.5, alpha=0.8,
            label='Casado-Alessi 1')

# overlay red X on bad ones
bad_good = good & bad_mask
if bad_good.any():
    plt.scatter(color_cl[bad_good], g_mag_cl[bad_good],
                marker='x', s=80, linewidth=2,
                color='red', label='Likely non-members')

# Isochrones
colors = plt.cm.viridis(np.linspace(0, 1, len(ages_gyr)))
for i, age_gyr in enumerate(ages_gyr):
    logage = np.log10(age_gyr * 1e9)
    iso = mist.isochrone(logage, feh)
    
    col_fit = (iso['BP_mag'] - iso['RP_mag']) + (K_BP - K_RP) * EBV_GLOBAL
    mag_fit = iso['G_mag'] + DM_GLOBAL + K_G * EBV_GLOBAL
    
    label = f"{age_gyr:.2f} Gyr"
    plt.plot(col_fit, mag_fit, color=colors[i], lw=2.5, label=label)

plt.gca().invert_yaxis()
plt.xlim(0.0, 2.5)
plt.ylim(np.nanmax(g_mag_cl[good])+1, np.nanmin(g_mag_cl[good])-1)
plt.xlabel(r'$G_{\rm BP} - G_{\rm RP}$  (mag)')
plt.ylabel(r'$G$  (mag)')
plt.title('Casado-Alessi 1 – MIST Isochrones')
plt.legend(fontsize=9, loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Holoviews not imported. Some visualizations will not be available.
PyMultiNest not imported.  MultiNest fits will not work.


FileNotFoundError: [Errno 2] No such file or directory: 'final_reordered.csv'